In [0]:
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
display(dbutils.fs.ls(f'{bronze_path}'))

In [0]:
display(dbutils.fs.ls(f'{bronze_path}/brands/'))

In [0]:
%sql
-- brands
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_brands
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/brands/'


In [0]:
"""
ETL com tabela temporária, geralmente não se cria tabelas físicas para a camada Bronze. Ela é utilizada para o refinamento e daí sim criar tabelas físicas na camada Silva
"""

In [0]:
# Criando tabela temporária - Para ser usada na próxima cadmada (camada Silver)
(spark.read.format('delta')
    .load(f'{bronze_path}/customers/')
    .createOrReplaceTempView('tmp_customers')
 )


In [0]:
%sql
SELECT * FROM tmp_customers

In [0]:
# Criando várias tabelas temporárias 
bronze_map = {
    'tmp_bronze_brands': f'{bronze_path}/brands/',
    'tmp_bronze_categories': f'{bronze_path}/categories/',
    'tmp_bronze_customers': f'{bronze_path}/customers/',
    'tmp_bronze_order_items': f'{bronze_path}/order_items/',
    'tmp_bronze_orders': f'{bronze_path}/orders/',
    'tmp_bronze_products': f'{bronze_path}/products/',
    'tmp_bronze_staffs': f'{bronze_path}/staffs/',
    'tmp_bronze_stocks': f'{bronze_path}/stocks/',
    'tmp_bronze_stores': f'{bronze_path}/stores/'
}
for view_name, path in bronze_map.items():
    (spark.read.format('delta')
    .load(path)
    .createOrReplaceTempView(view_name)
) 

In [0]:
%sql
-- Executando tabelas temporárias
SELECT * FROM tmp_bronze_brands;
-- SELECT * FROM tmp_bronze_categories;
-- SELECT * FROM tmp_bronze_customers;
-- SELECT * FROM tmp_bronze_order_items;
-- SELECT * FROM tmp_bronze_orders;
-- SELECT * FROM tmp_bronze_products;
-- SELECT * FROM tmp_bronze_staffs;
-- SELECT * FROM tmp_bronze_stores;



In [0]:
%sql
-- Verificando a descrição da tabela temporária 
DESCRIBE TABLE tmp_bronze_brands

In [0]:
%sql
SHOW TABLES IN bikestore.logistics

Exemplo de criar tabelas físicas

In [0]:
%sql
-- BRANDS
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_brands
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/brands/';

-- CATEGORIES
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_categories
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/categories/';

-- CUSTOMERS
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_customers
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/customers/';

-- ORDER_ITEMS
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_order_items
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/order_items/';

-- ORDERS
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_orders
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/orders/';

-- PRODUCTS
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_products
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/products/';

-- STAFFS
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_staffs
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/staffs/';

-- STOCKS
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_stocks
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/stocks/';

-- STORES
CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_stores
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/stores/';



Criando uma camada Silver de teste

In [0]:
%sql
SELECT * FROM tmp_bronze_products LIMIT 10

In [0]:
%sql
SELECT * FROM tmp_bronze_categories

In [0]:
%sql
DESCRIBE tmp_bronze_products

In [0]:
%sql
SELECT
    p.product_id,
    p.product_name,
    p.brand_id,
    p.category_id,
    p.model_year,
    p.list_price
FROM tmp_bronze_products AS p

In [0]:
%sql
DESCRIBE tmp_bronze_categories

In [0]:
%sql
SELECT
    p.product_id,
    p.product_name,
    p.brand_id,
    -- p.category_id AS category_id_produto,
    -- c.category_id AS category_id_categoria,
    c.category_name,
    p.model_year,
    p.list_price
FROM tmp_bronze_products AS p
LEFT JOIN tmp_bronze_categories AS c ON p.category_id = c.category_id


In [0]:
df_product_silver = spark.sql("""
SELECT
    p.product_id,
    p.product_name,
    p.brand_id,
    -- p.category_id AS category_id_produto,
    -- c.category_id AS category_id_categoria,
    c.category_name,
    p.model_year,
    p.list_price
FROM tmp_bronze_products AS p
LEFT JOIN tmp_bronze_categories AS c ON p.category_id = c.category_id                  
""")

In [0]:
display(df_product_silver )

## Salvando Tabela de Teste na camada Silver

In [0]:
# Salvar em parquet como Delta tabela de teste na camada Silver
df_product_silver.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .save(f'{silver_path}/product_teste')



In [0]:
%sql
/* Criar tabela física de teste este comando precisa rodar só 1 vez, pois quando os arquivos forem atualizados a tabela refletirá o resultado (Vai obter o log mais recente e trazer dados do(s) arquivo(s) atualizado)*/
CREATE TABLE IF NOT EXISTS bikestore.logistics.product_teste
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/product_teste'



In [0]:
%sql
SELECT * FROM bikestore.logistics.product_teste;